# Proust Attention Machine -- Colab Training

Train a character-level transformer on Proust's *En busca del tiempo perdido* (Spanish).
All 7 volumes (~7M chars) extracted from clean .mobi ebooks -- no OCR noise.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

## Checklist
1. Set runtime to **T4 GPU**
2. Run all cells in order
3. Checkpoints save to Google Drive (mounted automatically)
4. After training, download `best.pt` from Drive or use the generation cell

## Step 1 -- GPU Check & Google Drive Mount

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Mount Google Drive to persist checkpoints across sessions
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint directory on Drive (survives runtime disconnects)
CHECKPOINT_DIR = '/content/drive/MyDrive/proust-attention/checkpoints'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will save to: {CHECKPOINT_DIR}")

In [ ]:
# Keep session alive -- simulates a click every 60s to prevent idle disconnect
# Run this cell once; it stays active in the background for the entire session
from IPython.display import Javascript, display

display(Javascript('''
function keepAlive() {
    document.querySelector("colab-toolbar-button#connect").click();
    console.log("keep-alive ping", new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
console.log("Keep-alive started (every 60s)");
'''))
print("Keep-alive running. Session will stay active as long as this tab is open.")

## Step 2 -- Clone Repo

**Private repo?** You need a GitHub Personal Access Token:
1. Go to [github.com/settings/tokens](https://github.com/settings/tokens?type=beta)
2. *Generate new token (fine-grained)* > select only the `proust-attention` repo > Permission: Contents = Read
3. Copy the token -- the cell below will ask for it securely

In [ ]:
import os
from getpass import getpass

GITHUB_USER = 'GonorAndres'
BRANCH = '7feb'
REPO_DIR = '/content/proust-attention'

# Prompt for token (input is hidden, never printed)
TOKEN = getpass("GitHub token (paste and press Enter): ")

REPO_URL = f'https://{TOKEN}@github.com/{GITHUB_USER}/proust-attention.git'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

# Clear token from memory
del TOKEN, REPO_URL

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

## Step 3 -- Verify Corpus & Model

In [ ]:
import json
from pathlib import Path

corpus_path = Path('data/processed/proust_corpus.txt')
vocab_path = Path('data/processed/vocab.json')

assert corpus_path.exists(), "Corpus not found! Check the clone."
assert vocab_path.exists(), "Vocab not found! Check the clone."

corpus = open(corpus_path, encoding='utf-8').read()
vocab = json.load(open(vocab_path))

print(f"Corpus: {len(corpus):,} chars ({len(corpus)/1e6:.1f} MB)")
print(f"Vocab size: {vocab['vocab_size']} characters")
print(f"\nFirst 300 chars:\n{corpus[:300]}")
print(f"\nMiddle sample:\n{corpus[len(corpus)//2:len(corpus)//2+300]}")

# Verify no metadata leakage from ebook front/back matter
for bad in ['Librodot', 'ePub base', 'Titivillus', 'MARCEL PROUST (1871']:
    assert corpus.count(bad) == 0, f"LEAK: found '{bad}' in corpus!"
print("\nAll quality checks passed.")

In [ ]:
# Quick model parameter check
import sys
sys.path.insert(0, REPO_DIR)

from src.model_torch import Transformer, CONFIG

test_model = Transformer(vocab_size=vocab['vocab_size'], **{k: CONFIG[k] for k in ['d_model', 'n_heads', 'n_layers', 'd_ff', 'max_seq_len', 'dropout']})
n_params = test_model.count_parameters()
print(f"Model config: {CONFIG}")
print(f"Model parameters: {n_params:,}")
del test_model  # free memory before training

## Step 4 -- Train

Recommended settings for T4 GPU (15 GB VRAM):
- `batch_size=64` -- saturates the T4 without OOM
- `context_length=256` -- Proust's long sentences need room
- `epochs=50` -- full training run (~7M char corpus, may benefit from more epochs)

**Resuming a previous session**: set `EPOCHS` to the **remaining** epochs you want
(e.g. did 1 epoch already, want 50 total → set `EPOCHS = 49`).
The cell below auto-detects `best.pt` on Drive and passes `--resume` automatically.

In [ ]:
# ============================================================
# TRAINING CONFIGURATION -- edit these as needed
# ============================================================
EPOCHS = 49        # remaining epochs (total wanted - already done)
BATCH_SIZE = 64
CONTEXT_LENGTH = 256

In [ ]:
import os

best_pt = os.path.join(CHECKPOINT_DIR, 'best.pt')

# Auto-detect checkpoint on Drive and resume if found
if os.path.exists(best_pt):
    ckpt_info = torch.load(best_pt, map_location='cpu', weights_only=False)
    done_epochs = ckpt_info.get('epoch', '?')
    done_val    = ckpt_info.get('val_loss', '?')
    print(f"Found checkpoint: epoch {done_epochs}, val_loss {done_val:.4f}")
    print(f"Resuming training for {EPOCHS} more epochs...")
    resume_flag = f'--resume {best_pt}'
else:
    print("No checkpoint found -- starting from scratch")
    resume_flag = ''

!python src/train.py \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --context-length {CONTEXT_LENGTH} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    {resume_flag} \
    --device cuda

## Step 5 -- Generate Text from Trained Model

In [ ]:
best_pt = os.path.join(CHECKPOINT_DIR, 'best.pt')

if os.path.exists(best_pt):
    !python src/generate.py \
        --checkpoint {best_pt} \
        --prompt "Mucho tiempo he estado" \
        --length 500 \
        --temperature 0.8 \
        --top-k 40 \
        --device cuda
else:
    print(f"No best.pt found at {best_pt}")
    print("Training may not have completed a full epoch yet.")

In [ ]:
# Try different temperatures to see the effect
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n{'='*60}")
    print(f"Temperature: {temp}")
    print(f"{'='*60}")
    !python src/generate.py \
        --checkpoint {best_pt} \
        --prompt "Mucho tiempo" \
        --length 300 \
        --temperature {temp} \
        --top-k 40 \
        --device cuda

## Step 6 -- Download Checkpoint (Optional)

Your checkpoints are already on Google Drive. To download `best.pt` to your local machine:

In [ ]:
# Download best.pt directly from Colab
from google.colab import files
if os.path.exists(best_pt):
    files.download(best_pt)
else:
    print("No best.pt to download yet.")